# LeetCode 87 — Scramble String (Hard) bằng Python

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def isScramble(self, s1: str, s2: str) -> bool:

```

**Mục tiêu sau khi học xong:**

1. Hiểu đúng định nghĩa "xáo trộn" (scramble) của đề: **cắt** chuỗi thành 2 phần, **có thể đổi chỗ** hai phần, rồi làm tiếp **đệ quy** trên từng phần — và nhìn nó như một **cây chia đôi**.
2. Học **đệ quy (recursion)** từ căn bản, rồi viết **Cách 1 — sinh tất cả chuỗi scramble** của `s1` để thấy định nghĩa "chạy" ra sao (chỉ dùng được với chuỗi rất ngắn).
3. Rút ra **insight**: chỉ cần nhìn **nhát cắt đầu tiên** → viết **Cách 2 — đệ quy kiểm tra trực tiếp** (thử mọi điểm cắt, 2 trường hợp: không đổi chỗ / đổi chỗ).
4. Thêm **cắt tỉa (pruning)** bằng `sorted` / `Counter` → **Cách 3**, và tự tay tìm ra input làm cách này vẫn chậm.
5. Học **memoization** và viết **Cách 4 — đệ quy + memo** với trạng thái `(i1, i2, length)`.
6. Viết **Cách 5 — DP 3 chiều bottom-up `dp[length][i][j]`**, in bảng từng tầng cho `"great"` / `"rgeat"`.
7. Nộp được `class Solution`, kiểm thử chéo các cách trên input ngẫu nhiên, nhận ra pattern **"chia đôi tại mọi điểm cắt" (interval / partition DP)** để dùng lại cho bài khác.

## 1. Đề bài

Ta có thể **xáo trộn (scramble)** một chuỗi `s` để được chuỗi `t` bằng thuật toán sau:

1. Nếu độ dài chuỗi là 1 thì dừng.
2. Nếu độ dài chuỗi lớn hơn 1 thì làm như sau:
   - **Chia** chuỗi thành hai chuỗi con **khác rỗng** tại một vị trí ngẫu nhiên, tức là nếu chuỗi là `s` thì chia thành `x` và `y` với `s = x + y`.
   - **Ngẫu nhiên quyết định** có hoán đổi hai chuỗi con hay giữ nguyên thứ tự. Tức là sau bước này, `s` có thể trở thành `s = x + y` hoặc `s = y + x`.
   - **Áp dụng bước 1 một cách đệ quy** lên từng chuỗi con `x` và `y`.

Cho hai chuỗi `s1` và `s2` **có cùng độ dài**, trả về `true` nếu `s2` là một chuỗi xáo trộn (scrambled string) của `s1`, ngược lại trả về `false`.

**Ví dụ 1**

```text
Input: s1 = "great", s2 = "rgeat"
Output: true
Giải thích: Một kịch bản có thể xảy ra trên s1:
"great" --> "gr/eat"            // chia tại một vị trí ngẫu nhiên.
"gr/eat" --> "gr/eat"           // quyết định ngẫu nhiên: KHÔNG đổi chỗ hai chuỗi con.
"gr/eat" --> "g/r / e/at"       // áp dụng đệ quy lên cả hai chuỗi con: chia mỗi chuỗi tại một vị trí ngẫu nhiên.
"g/r / e/at" --> "r/g / e/at"   // quyết định: ĐỔI CHỖ chuỗi con thứ nhất, giữ nguyên chuỗi con thứ hai.
"r/g / e/at" --> "r/g / e/ a/t" // lại áp dụng đệ quy, chia "at" thành "a/t".
"r/g / e/ a/t" --> "r/g / e/ a/t" // quyết định: giữ nguyên thứ tự cả hai chuỗi con.
Thuật toán dừng, chuỗi kết quả là "rgeat", chính là s2.
Vì có một kịch bản biến s1 thành s2, ta trả về true.
```

**Ví dụ 2**

```text
Input: s1 = "abcde", s2 = "caebd"
Output: false
```

**Ví dụ 3**

```text
Input: s1 = "a", s2 = "a"
Output: true
```

**Constraints (ràng buộc):**

- `s1.length == s2.length`
- `1 <= s1.length <= 30`
- `s1` và `s2` chỉ gồm chữ cái tiếng Anh viết thường.

### Đọc đề chậm lại: đề hỏi gì?

Đề **không** bắt ta xáo trộn ngẫu nhiên. Đề hỏi: **trong tất cả các cách** "cắt + có thể đổi chỗ + làm tiếp", **có tồn tại ít nhất một cách** biến `s1` thành `s2` không?

Ba điều cần khắc ghi ngay từ đầu:

- Mỗi nhát cắt chia thành **2 phần khác rỗng**. Với chuỗi dài `n`, điểm cắt `i` chạy từ `1` tới `n - 1` (phần trái dài `i`, phần phải dài `n - i`).
- Sau khi cắt, hai phần **chỉ được đổi chỗ với nhau**, không được trộn chữ của phần này vào phần kia. Đây là lý do "scramble" **không** giống "hoán vị tùy ý" (anagram) — ta sẽ thấy phản ví dụ cụ thể ở mục 4.
- Luôn được phép "không đổi chỗ" ở mọi bước, nên `s1` **luôn** là scramble của chính nó (ví dụ 3).

Cell dưới lưu 3 ví dụ của đề vào một list để dùng lại cho mọi cách giải phía sau.

In [1]:
EXAMPLES = [
    ("great", "rgeat", True),
    ("abcde", "caebd", False),
    ("a", "a", True),
]

for s1, s2, expected in EXAMPLES:
    print(f"s1={s1!r:8} s2={s2!r:8} -> kỳ vọng {expected}   (cùng độ dài: {len(s1) == len(s2)})")

s1='great'  s2='rgeat'  -> kỳ vọng True   (cùng độ dài: True)
s1='abcde'  s2='caebd'  -> kỳ vọng False   (cùng độ dài: True)
s1='a'      s2='a'      -> kỳ vọng True   (cùng độ dài: True)


## 2. Kiến thức nền

### 2.1. Cắt chuỗi bằng slicing: `s[:i]` và `s[i:]`

Trong Python, `s[a:b]` lấy các ký tự từ chỉ số `a` **đến trước** `b`. Khi bỏ trống:

- `s[:i]` = `i` ký tự **đầu** (chỉ số `0 .. i-1`).
- `s[i:]` = phần **còn lại** từ chỉ số `i` tới hết (dài `n - i`).
- `s[n-i:]` = `i` ký tự **cuối**. Ví dụ `n = 5`, `i = 2` → `s[3:]` là 2 ký tự cuối.

Vì đề yêu cầu hai phần **khác rỗng**, điểm cắt hợp lệ là `i = 1, 2, ..., n-1` — trong Python viết `range(1, n)`. Nếu lỡ cho `i = 0` thì `s[:0] = ""` (rỗng), vi phạm đề và (như ta sẽ thấy ở mục Lỗi thường gặp) gây đệ quy vô hạn.

In [2]:
s = "great"
n = len(s)
for i in range(1, n):
    x, y = s[:i], s[i:]
    print(f"i={i}: x={x!r:7} y={y!r:7}  giữ nguyên -> {x + y!r}   đổi chỗ -> {y + x!r}")

print("\n2 ký tự cuối của 'rgeat':", "rgeat"[5 - 2:])

i=1: x='g'     y='reat'   giữ nguyên -> 'great'   đổi chỗ -> 'reatg'
i=2: x='gr'    y='eat'    giữ nguyên -> 'great'   đổi chỗ -> 'eatgr'
i=3: x='gre'   y='at'     giữ nguyên -> 'great'   đổi chỗ -> 'atgre'
i=4: x='grea'  y='t'      giữ nguyên -> 'great'   đổi chỗ -> 'tgrea'

2 ký tự cuối của 'rgeat': at


### 2.2. Nhìn một lần xáo trộn như một **cây chia đôi**

Kịch bản ở Ví dụ 1 thực chất là một **cây nhị phân**: mỗi nút là một nhát cắt (trái | phải) kèm một quyết định "có đổi chỗ hay không"; lá là các mẩu chuỗi không cắt nữa.

Ta biểu diễn cây bằng **tuple lồng nhau** cho dễ gõ:

- Một **lá** là một chuỗi, ví dụ `"g"`. (Một lá dài hơn 1, ví dụ `"at"`, nghĩa là "cắt bên trong nhưng không bao giờ đổi chỗ" — kết quả vẫn y nguyên `"at"`.)
- Một **nút** là `(trái, phải, swap)` với `swap = True` nghĩa là đổi chỗ hai phần.

Kịch bản đề bài: `"great"` cắt thành `"gr" | "eat"` (không đổi); `"gr"` cắt `"g" | "r"` (**đổi**); `"eat"` cắt `"e" | "at"` (không đổi); `"at"` cắt `"a" | "t"` (không đổi).

Hàm `apply_tree` "chạy" cây từ dưới lên: tính kết quả của con trái, con phải, rồi ghép theo `swap`. Hàm `tree_source` ghép lá theo thứ tự gốc (bỏ qua `swap`) để lấy lại `s1`. Hàm `print_tree` vẽ cây bằng thụt lề.

In [3]:
def apply_tree(tree):
    # Trả về chuỗi KẾT QUẢ sau khi xáo trộn theo cây.
    if isinstance(tree, str):          # lá
        return tree
    left, right, swap = tree
    a, b = apply_tree(left), apply_tree(right)
    return b + a if swap else a + b


def tree_source(tree):
    # Trả về chuỗi GỐC (đọc lá từ trái sang phải, bỏ qua swap).
    if isinstance(tree, str):
        return tree
    left, right, _ = tree
    return tree_source(left) + tree_source(right)


def print_tree(tree, indent=""):
    if isinstance(tree, str):
        print(indent + repr(tree))
        return
    left, right, swap = tree
    note = "ĐỔI CHỖ" if swap else "giữ nguyên"
    print(f"{indent}{tree_source(tree)!r} = {tree_source(left)!r} | {tree_source(right)!r}  [{note}] -> {apply_tree(tree)!r}")
    print_tree(left, indent + "    ")
    print_tree(right, indent + "    ")


great_tree = (("g", "r", True), ("e", ("a", "t", False), False), False)
print_tree(great_tree)
print("\nChuỗi gốc :", tree_source(great_tree))
print("Kết quả   :", apply_tree(great_tree))

'great' = 'gr' | 'eat'  [giữ nguyên] -> 'rgeat'
    'gr' = 'g' | 'r'  [ĐỔI CHỖ] -> 'rg'
        'g'
        'r'
    'eat' = 'e' | 'at'  [giữ nguyên] -> 'eat'
        'e'
        'at' = 'a' | 't'  [giữ nguyên] -> 'at'
            'a'
            't'

Chuỗi gốc : great
Kết quả   : rgeat


Đọc cây từ dưới lên: `"a" | "t"` giữ nguyên → `"at"`; `"e" | "at"` giữ nguyên → `"eat"`; `"g" | "r"` **đổi chỗ** → `"rg"`; cuối cùng `"gr" | "eat"` giữ nguyên → `"rg" + "eat" = "rgeat"`.

Nhận xét quan trọng cho cả bài: **ở mỗi nút, hai nửa chỉ có thể đổi chỗ cho nhau, còn chữ cái bên trong mỗi nửa thì "ở yên trong nửa đó"**. Câu hỏi của đề tương đương: *có tồn tại một cây như vậy biến `s1` thành `s2` không?*

### 2.3. Đệ quy (recursion) từ căn bản

**Đệ quy** là khi một hàm **tự gọi lại chính nó** trên một bài toán **nhỏ hơn**. Mỗi hàm đệ quy cần 2 phần:

1. **Base case (trường hợp cơ sở)**: bài toán đủ nhỏ để trả lời ngay, không gọi tiếp. Thiếu phần này → gọi mãi không dừng.
2. **Recursive case (bước đệ quy)**: chia bài toán thành bài nhỏ hơn, gọi lại hàm, rồi ghép kết quả.

Ví dụ mini: đảo ngược chuỗi.

- Base case: chuỗi dài ≤ 1 thì đảo ngược chính là nó.
- Bước đệ quy: `reverse("abc") = reverse("bc") + "a"`.

Hai hàm `apply_tree` / `tree_source` ở trên cũng là đệ quy: lá là base case, nút gọi lại hàm cho 2 con. Cell dưới thêm tham số `depth` để **in thụt lề theo độ sâu** — đây chính là cách ta sẽ trace các lời giải đệ quy phía sau.

In [4]:
def reverse_rec(s, depth=0):
    pad = "    " * depth
    print(f"{pad}reverse({s!r})")
    if len(s) <= 1:                       # base case
        print(f"{pad}  -> {s!r} (base case)")
        return s
    result = reverse_rec(s[1:], depth + 1) + s[0]    # bước đệ quy
    print(f"{pad}  -> {result!r}")
    return result


reverse_rec("abc")

reverse('abc')
    reverse('bc')
        reverse('c')
          -> 'c' (base case)
      -> 'cb'
  -> 'cba'


'cba'

Đọc output: lời gọi `reverse('abc')` phải **chờ** `reverse('bc')`, lời gọi đó lại chờ `reverse('c')`. Khi chạm base case, kết quả "trả ngược" lên từng tầng: `'c'` → `'cb'` → `'cba'`. Python lưu các lời gọi đang chờ trong **call stack (ngăn xếp lời gọi)**; độ sâu tối đa mặc định khoảng 1000 tầng — với bài này độ dài ≤ 30 nên độ sâu không phải vấn đề.

### 2.4. Mô phỏng đúng thuật toán của đề bằng `random`

Để có nhiều cặp `(s1, s2)` mà **chắc chắn** trả lời `True` (dùng cho test sau này), ta cài đúng 3 bước của đề: cắt ngẫu nhiên, đổi chỗ ngẫu nhiên, làm tiếp đệ quy. `random.Random(seed)` tạo bộ sinh số ngẫu nhiên riêng với **hạt giống (seed)** cố định, nên mỗi lần chạy notebook đều ra cùng kết quả.

- `rng.randint(1, n - 1)` chọn điểm cắt hợp lệ (khác rỗng).
- `rng.random() < 0.5` là "tung đồng xu" quyết định đổi chỗ.

In [5]:
import random

def random_scramble(s, rng):
    if len(s) == 1:                      # bước 1 của đề: dài 1 thì dừng
        return s
    i = rng.randint(1, len(s) - 1)       # chia thành x, y khác rỗng
    x, y = s[:i], s[i:]
    x, y = random_scramble(x, rng), random_scramble(y, rng)  # đệ quy trên từng phần
    return y + x if rng.random() < 0.5 else x + y            # đổi chỗ hoặc giữ nguyên


rng = random.Random(87)
print([random_scramble("great", rng) for _ in range(6)])

['great', 'gerat', 'reatg', 'targe', 'grtea', 'gerta']


## 3. Cách 1 — Sinh **tất cả** chuỗi scramble của `s1` (trực giác nhất)

Người mới thường nghĩ: "Cứ liệt kê hết mọi chuỗi có thể tạo ra từ `s1`, rồi xem `s2` có nằm trong đó không."

Định nghĩa đệ quy của tập `ALL(s)` (mọi chuỗi scramble của `s`):

- Base case: `ALL("g") = {"g"}`.
- Bước đệ quy: với mỗi điểm cắt `i`, gọi `x = s[:i]`, `y = s[i:]`. Với mỗi `a` thuộc `ALL(x)` và mỗi `b` thuộc `ALL(y)`, ta có 2 kết quả: `a + b` (giữ nguyên) và `b + a` (đổi chỗ).

Nhiều kịch bản khác nhau có thể cho ra **cùng một chuỗi** (ví dụ `"abc"` có thể đạt được bằng nhiều cây), nên ta cần một cấu trúc **tự loại trùng**: đó là `set`.

### Kiến thức nền: `set` (tập hợp)

`set` lưu các phần tử **không trùng nhau**: `add(x)` thêm phần tử (đã có thì bỏ qua), `x in st` kiểm tra tồn tại (trung bình O(1)), `len(st)` đếm số phần tử khác nhau.

In [6]:
st = set()
st.add("abc")
st.add("bca")
st.add("abc")          # trùng -> không thêm
print(st, len(st))
print("bca" in st, "cab" in st)

{'bca', 'abc'} 2
True False


In [7]:
def all_scrambles(s):
    # Trả về set mọi chuỗi scramble của s (đệ quy thuần, chỉ dùng cho s rất ngắn).
    if len(s) == 1:
        return {s}
    result = set()
    for i in range(1, len(s)):
        for a in all_scrambles(s[:i]):
            for b in all_scrambles(s[i:]):
                result.add(a + b)    # giữ nguyên
                result.add(b + a)    # đổi chỗ
    return result


def is_scramble_generate(s1, s2):
    return s2 in all_scrambles(s1)


print("ALL('ab')  =", sorted(all_scrambles("ab")))
print("ALL('abc') =", sorted(all_scrambles("abc")))
for s1, s2, expected in EXAMPLES:
    print(s1, s2, "->", is_scramble_generate(s1, s2), "(kỳ vọng", expected, ")")

ALL('ab')  = ['ab', 'ba']
ALL('abc') = ['abc', 'acb', 'bac', 'bca', 'cab', 'cba']
great rgeat -> True (kỳ vọng True )
abcde caebd -> False (kỳ vọng False )
a a -> True (kỳ vọng True )


### Quan sát quan trọng: scramble **không phải** hoán vị tùy ý

Với `"abc"`, ta sinh được đủ 6 hoán vị. Dễ tưởng rằng "scramble = sắp xếp lại chữ cái kiểu gì cũng được". Nhưng với `"abcd"` (4 chữ khác nhau, có 4! = 24 hoán vị), hãy đếm thử:

In [8]:
from itertools import permutations

all4 = all_scrambles("abcd")
perms4 = {"".join(p) for p in permutations("abcd")}
print("Số hoán vị của 'abcd'  :", len(perms4))
print("Số chuỗi scramble      :", len(all4))
print("Hoán vị KHÔNG đạt được :", sorted(perms4 - all4))

for n in range(1, 8):
    word = "abcdefg"[:n]
    print(f"n={n}  '{word}': {len(all_scrambles(word))} chuỗi scramble")

Số hoán vị của 'abcd'  : 24
Số chuỗi scramble      : 22
Hoán vị KHÔNG đạt được : ['bdac', 'cadb']
n=1  'a': 1 chuỗi scramble
n=2  'ab': 2 chuỗi scramble
n=3  'abc': 6 chuỗi scramble
n=4  'abcd': 22 chuỗi scramble
n=5  'abcde': 90 chuỗi scramble
n=6  'abcdef': 394 chuỗi scramble
n=7  'abcdefg': 1806 chuỗi scramble


Chỉ có **22/24** hoán vị đạt được: `"bdac"` và `"cadb"` **không** là scramble của `"abcd"` dù có cùng bộ chữ cái. Thử lý giải `"bdac"`: nhát cắt đầu tiên của `"abcd"` phải chia thành một khối liền, ví dụ `"a" | "bcd"`, `"ab" | "cd"`, `"abc" | "d"`; sau đó các chữ của mỗi khối phải nằm **liền nhau** trong kết quả. Nhưng trong `"bdac"`, không có cách tách thành 2 đoạn liền mà mỗi đoạn là đúng một khối như trên (`"b"|"dac"` → `{b}` không phải `{a}` hay `{d}`; `"bd"|"ac"` → `{b,d}` không phải `{a,b}` hay `{c,d}`; `"bda"|"c"` → `{c}` không phải `{a}`/`{d}`). Vậy câu trả lời là `False`.

Bài học: **cùng bộ chữ cái là điều kiện CẦN, không phải điều kiện ĐỦ.** Ta sẽ dùng điều này để cắt tỉa ở Cách 3.

### Cách 1 vướng ở đâu?

Số chuỗi scramble của `n` chữ cái khác nhau là `1, 2, 6, 22, 90, 394, 1806, ...` (dãy **số Schröder lớn** trong toán tổ hợp), tăng gần **6 lần** mỗi khi thêm 1 ký tự. Cell dưới tính dãy này bằng công thức truy hồi đã biết để ước lượng cho `n = 30` — không sinh chuỗi nào cả.

In [9]:
# Số Schröder lớn: S[0]=1, S[1]=2, (k+1)*S[k] = 3*(2k-1)*S[k-1] - (k-2)*S[k-2]
# Số chuỗi scramble của n chữ cái khác nhau = S[n-1]
S = [1, 2]
for k in range(2, 30):
    S.append((3 * (2 * k - 1) * S[k - 1] - (k - 2) * S[k - 2]) // (k + 1))

print("n = 1..7 :", S[:7])
print(f"n = 30   : {S[29]:,} chuỗi (~{S[29]:.1e})")

n = 1..7 : [1, 2, 6, 22, 90, 394, 1806]
n = 30   : 79,228,031,819,993,134,650 chuỗi (~7.9e+19)


Với `n = 30`, cần sinh khoảng `10^20` chuỗi — không máy nào chạy nổi, càng không đủ bộ nhớ để chứa cái `set` đó.

**Độ phức tạp Cách 1:** thời gian và bộ nhớ đều cỡ **số chuỗi scramble** (theo cấp số mũ, gần `5.8^n`) nhân thêm `O(n)` để tạo mỗi chuỗi. Chỉ dùng để **hiểu định nghĩa** và làm "trọng tài" kiểm tra cho chuỗi ngắn (`n ≤ 6`).

**Nút thắt:** ta đang **sinh hết** rồi mới tra. Câu hỏi chỉ cần **có/không** cho một `s2` cụ thể — hãy để chính `s2` dẫn đường.

## 4. Insight: chỉ cần nhìn **nhát cắt đầu tiên**

Giả sử `s2` là scramble của `s1` (dài `n`). Nhìn **gốc cây**: `s1` bị cắt tại `i` thành `x = s1[:i]` (dài `i`) và `y = s1[i:]` (dài `n - i`). Sau đó `x` biến thành `x'`, `y` biến thành `y'` (mỗi cái là scramble của phần tương ứng, **độ dài không đổi**). Có đúng 2 khả năng:

| Trường hợp | `s2` bằng | Vậy phải có |
|---|---|---|
| **Không đổi chỗ** | `x' + y'` | `s2[:i]` là scramble của `s1[:i]` **và** `s2[i:]` là scramble của `s1[i:]` |
| **Đổi chỗ** | `y' + x'` | `s2[n-i:]` (i ký tự **cuối**) là scramble của `s1[:i]` **và** `s2[:n-i]` là scramble của `s1[i:]` |

Mấu chốt nằm ở **độ dài**: `x'` dài đúng `i`, nên khi không đổi chỗ nó chiếm `i` ký tự **đầu** của `s2`; khi đổi chỗ nó bị đẩy ra **cuối**, chiếm `i` ký tự **cuối** của `s2`.

Ngược lại, nếu với một `i` nào đó một trong hai dòng của bảng đúng, ta ghép hai cây con lại thành cây cho `(s1, s2)`. Vậy:

> `s2` là scramble của `s1`  ⇔  (`n == 1` và `s1 == s2`) **hoặc** tồn tại `i ∈ [1, n-1]` thỏa "không đổi chỗ" hoặc "đổi chỗ".

Ví dụ số với `s1 = "great"`, `s2 = "rgeat"`, `i = 2`: `x = "gr"`, `y = "eat"`.
- Không đổi chỗ: so `"gr"` với `s2[:2] = "rg"` và `"eat"` với `s2[2:] = "eat"`.
- Đổi chỗ: so `"gr"` với `s2[3:] = "at"` và `"eat"` với `s2[:3] = "rge"`.

Cell dưới in đủ các cặp cần so cho mọi `i`.

In [10]:
def show_subproblems(s1, s2):
    n = len(s1)
    for i in range(1, n):
        keep = (s1[:i], s2[:i], s1[i:], s2[i:])
        swap = (s1[:i], s2[n - i:], s1[i:], s2[:n - i])
        print(f"i={i}  không đổi: ({keep[0]!r},{keep[1]!r}) & ({keep[2]!r},{keep[3]!r})")
        print(f"     đổi chỗ  : ({swap[0]!r},{swap[1]!r}) & ({swap[2]!r},{swap[3]!r})")


show_subproblems("great", "rgeat")

i=1  không đổi: ('g','r') & ('reat','geat')
     đổi chỗ  : ('g','t') & ('reat','rgea')
i=2  không đổi: ('gr','rg') & ('eat','eat')
     đổi chỗ  : ('gr','at') & ('eat','rge')
i=3  không đổi: ('gre','rge') & ('at','at')
     đổi chỗ  : ('gre','eat') & ('at','rg')
i=4  không đổi: ('grea','rgea') & ('t','t')
     đổi chỗ  : ('grea','geat') & ('t','r')


Nhìn dòng `i=2`, không đổi chỗ: cặp `("gr", "rg")` — một bài toán **nhỏ hơn** cùng kiểu (cắt `"g"|"r"`, đổi chỗ là xong), và cặp `("eat", "eat")` — hiển nhiên đúng. Vậy `"rgeat"` là scramble của `"great"`, khớp với cây ở mục 2.2.

Bài toán gốc kích thước `n` đã được quy về các bài toán **cùng dạng, nhỏ hơn**. Đó là tín hiệu rõ nhất để dùng **đệ quy**.

## 5. Cách 2 — Đệ quy kiểm tra trực tiếp

Dịch nguyên văn insight thành code:

- **Base case:** `len(s1) == 1` → trả về `s1 == s2`.
- **Bước đệ quy:** với mỗi `i` trong `range(1, n)`:
  - nếu `rec(s1[:i], s2[:i]) and rec(s1[i:], s2[i:])` → `True` (không đổi chỗ);
  - nếu `rec(s1[:i], s2[n-i:]) and rec(s1[i:], s2[:n-i])` → `True` (đổi chỗ).
- Thử hết mà không được → `False`.

Lưu ý **short-circuit**: với `A and B`, nếu `A` là `False` thì Python **không tính** `B` nữa. Nhờ vậy, khi nửa đầu đã sai thì nửa sau không bị gọi đệ quy vô ích. Và `return True` ngay khi tìm thấy một cách là đủ (đề chỉ hỏi "có tồn tại").

In [11]:
def is_scramble_recursive(s1, s2):
    n = len(s1)
    if n == 1:
        return s1 == s2
    for i in range(1, n):
        # Trường hợp 1: không đổi chỗ
        if is_scramble_recursive(s1[:i], s2[:i]) and is_scramble_recursive(s1[i:], s2[i:]):
            return True
        # Trường hợp 2: đổi chỗ
        if is_scramble_recursive(s1[:i], s2[n - i:]) and is_scramble_recursive(s1[i:], s2[:n - i]):
            return True
    return False


for s1, s2, expected in EXAMPLES:
    got = is_scramble_recursive(s1, s2)
    print(f"{s1!r} vs {s2!r} -> {got}")
    assert got == expected

'great' vs 'rgeat' -> True
'abcde' vs 'caebd' -> False
'a' vs 'a' -> True


### Trace Cách 2: cây lời gọi cho `"great"` / `"rgeat"`

Hàm `trace_recursive` giống hệt Cách 2 nhưng in mỗi lời gọi với thụt lề theo độ sâu, ghi rõ đang thử `i` nào, trường hợp nào, và kết quả trả về.

In [12]:
def trace_recursive(s1, s2, depth=0, tag="gốc"):
    pad = "  " * depth
    n = len(s1)
    if n == 1:
        print(f"{pad}[{tag}] {s1!r} vs {s2!r}: base case -> {s1 == s2}")
        return s1 == s2
    print(f"{pad}[{tag}] {s1!r} vs {s2!r}")
    for i in range(1, n):
        if trace_recursive(s1[:i], s2[:i], depth + 1, f"i={i} giữ") and \
           trace_recursive(s1[i:], s2[i:], depth + 1, f"i={i} giữ"):
            print(f"{pad}  => True (cắt i={i}, không đổi chỗ)")
            return True
        if trace_recursive(s1[:i], s2[n - i:], depth + 1, f"i={i} đổi") and \
           trace_recursive(s1[i:], s2[:n - i], depth + 1, f"i={i} đổi"):
            print(f"{pad}  => True (cắt i={i}, đổi chỗ)")
            return True
    print(f"{pad}  => False")
    return False


trace_recursive("great", "rgeat")

[gốc] 'great' vs 'rgeat'
  [i=1 giữ] 'g' vs 'r': base case -> False
  [i=1 đổi] 'g' vs 't': base case -> False
  [i=2 giữ] 'gr' vs 'rg'
    [i=1 giữ] 'g' vs 'r': base case -> False
    [i=1 đổi] 'g' vs 'g': base case -> True
    [i=1 đổi] 'r' vs 'r': base case -> True
    => True (cắt i=1, đổi chỗ)
  [i=2 giữ] 'eat' vs 'eat'
    [i=1 giữ] 'e' vs 'e': base case -> True
    [i=1 giữ] 'at' vs 'at'
      [i=1 giữ] 'a' vs 'a': base case -> True
      [i=1 giữ] 't' vs 't': base case -> True
      => True (cắt i=1, không đổi chỗ)
    => True (cắt i=1, không đổi chỗ)
  => True (cắt i=2, không đổi chỗ)


True

### Đọc trace bằng bảng

| Bước | Cặp đang xét | Điểm cắt / trường hợp | Kết quả | Vì sao |
|---|---|---|---|---|
| 1 | `"great"` vs `"rgeat"` | `i=1` giữ: `"g"` vs `"r"` | False | base case, khác chữ → short-circuit, bỏ qua nửa sau |
| 2 | | `i=1` đổi: `"g"` vs `"t"` | False | base case, khác chữ |
| 3 | | `i=2` giữ: `"gr"` vs `"rg"` | (đệ quy) | bài toán con dài 2 |
| 4 | `"gr"` vs `"rg"` | `i=1` giữ: `"g"` vs `"r"` | False | khác chữ |
| 5 | | `i=1` đổi: `"g"` vs `"g"`, `"r"` vs `"r"` | **True** | đổi chỗ khớp → `"gr"` ~ `"rg"` |
| 6 | `"great"` vs `"rgeat"` | `i=2` giữ, nửa sau: `"eat"` vs `"eat"` | **True** | cắt `i=1` rồi `i=1` trong `"at"`, đều giữ nguyên |
| 7 | | | **True** | cắt `i=2`, không đổi chỗ |

Để ý: để xác nhận `"eat"` ~ `"eat"`, Cách 2 vẫn phải đệ quy xuống tận từng chữ cái — dù hai chuỗi **giống hệt nhau**. Đây là chỗ đầu tiên ta sẽ tối ưu.

### Cách 2 vướng ở đâu? Đếm số lời gọi

Ví dụ trên trả lời nhanh vì tìm thấy `True` sớm. Trường hợp **xấu** là khi đáp án là `False`: hàm phải thử **mọi** `i` và **mọi** trường hợp. Ta đếm số lời gọi với họ input `s1 = "abcde..."`, `s2 = "bdac" + phần còn lại` (luôn `False` vì đoạn đầu `"bdac"` không đạt được — đã thấy ở mục 3).

Để đếm, ta dùng một list 1 phần tử `calls = [0]` và tăng `calls[0] += 1` ở đầu hàm (list là đối tượng thay đổi được, nên hàm bên trong sửa được giá trị mà không cần `global`).

In [13]:
def count_calls_recursive(s1, s2):
    calls = [0]

    def rec(a, b):
        calls[0] += 1
        n = len(a)
        if n == 1:
            return a == b
        for i in range(1, n):
            if rec(a[:i], b[:i]) and rec(a[i:], b[i:]):
                return True
            if rec(a[:i], b[n - i:]) and rec(a[i:], b[:n - i]):
                return True
        return False

    return rec(s1, s2), calls[0]


prev = None
for n in range(5, 12):
    s1 = "abcdefghijk"[:n]
    s2 = "bdac" + s1[4:]
    ans, calls = count_calls_recursive(s1, s2)
    ratio = f"x{calls / prev:.2f}" if prev else ""
    print(f"n={n:2}  {s1!r:14} vs {s2!r:14} -> {ans}, {calls:6} lời gọi {ratio}")
    prev = calls

n= 5  'abcde'        vs 'bdace'        -> False,    105 lời gọi 
n= 6  'abcdef'       vs 'bdacef'       -> False,    318 lời gọi x3.03
n= 7  'abcdefg'      vs 'bdacefg'      -> False,    967 lời gọi x3.04
n= 8  'abcdefgh'     vs 'bdacefgh'     -> False,   2943 lời gọi x3.04
n= 9  'abcdefghi'    vs 'bdacefghi'    -> False,   8949 lời gọi x3.04
n=10  'abcdefghij'   vs 'bdacefghij'   -> False,  27167 lời gọi x3.04
n=11  'abcdefghijk'  vs 'bdacefghijk'  -> False,  82317 lời gọi x3.03


Mỗi khi thêm 1 ký tự, số lời gọi **tăng khoảng 3 lần**. Ngoại suy tới `n = 30`: cỡ `82317 · 3^19 ≈ 10^14` lời gọi — hoàn toàn không chạy nổi.

**Độ phức tạp Cách 2:** thời gian **hàm mũ** theo `n` (đo được tăng ~3 lần mỗi ký tự trên họ input này; trường hợp xấu nhất còn tệ hơn), bộ nhớ `O(n^2)` cho call stack (độ sâu `≤ n`, mỗi tầng giữ các lát cắt dài `≤ n`).

Có **hai nguồn lãng phí** khác nhau:

1. **Thử những nhánh vô vọng**: ví dụ so `"ab"` với `"ca"` — chỉ cần đếm chữ cái là biết `False`, vậy mà Cách 2 vẫn đệ quy xuống. → Sửa bằng **cắt tỉa** (Cách 3).
2. **Giải đi giải lại cùng một bài toán con**: cùng một cặp `(a, b)` xuất hiện ở nhiều nhánh của cây lời gọi. → Sửa bằng **memo** (Cách 4).

## 6. Cách 3 — Đệ quy + cắt tỉa (pruning)

### Kiến thức nền: kiểm tra 2 chuỗi có cùng bộ chữ cái (anagram)

Hai chuỗi là **anagram** của nhau nếu chúng có cùng các chữ cái với cùng số lần xuất hiện. Hai cách kiểm tra:

- `sorted(a) == sorted(b)`: `sorted("caebd")` trả về list chữ cái đã sắp xếp `['a','b','c','d','e']`. Tốn `O(L log L)` với `L` là độ dài.
- `Counter(a) == Counter(b)`: `collections.Counter` đếm số lần xuất hiện của mỗi ký tự (một dict `ký tự -> số lần`). Tốn `O(L)`.

Vì scramble chỉ **sắp xếp lại** chữ cái, nếu `a` và `b` **không** phải anagram thì chắc chắn `False`. Nhưng như đã thấy, anagram **chưa đủ** để kết luận `True`.

In [14]:
from collections import Counter

print(sorted("caebd"))
print(Counter("aabcb"))
print("'ab' vs 'ca'    anagram?", sorted("ab") == sorted("ca"))
print("'abcd' vs 'bdac' anagram?", Counter("abcd") == Counter("bdac"),
      "| scramble?", is_scramble_recursive("abcd", "bdac"))

['a', 'b', 'c', 'd', 'e']
Counter({'a': 2, 'b': 2, 'c': 1})
'ab' vs 'ca'    anagram? False
'abcd' vs 'bdac' anagram? True | scramble? False


### Hai nhát cắt tỉa

Thêm vào đầu hàm đệ quy, **trước** vòng lặp thử điểm cắt:

1. `if s1 == s2: return True` — chuỗi giống hệt thì luôn là scramble (không đổi chỗ ở đâu cả). Cắt bỏ cả nhánh `"eat"` vs `"eat"` ở trace trên.
2. `if sorted(s1) != sorted(s2): return False` — khác bộ chữ cái thì chắc chắn sai.

Cắt tỉa (2) còn **thay luôn base case**: với `n == 1`, hoặc `s1 == s2` (nhánh 1) hoặc khác chữ (nhánh 2), nên không bao giờ tới vòng lặp với `n == 1`.

In [15]:
def is_scramble_pruned(s1, s2):
    if s1 == s2:                       # cắt tỉa 1: giống hệt
        return True
    if sorted(s1) != sorted(s2):       # cắt tỉa 2: khác bộ chữ cái (cũng lo luôn n == 1)
        return False
    n = len(s1)
    for i in range(1, n):
        if is_scramble_pruned(s1[:i], s2[:i]) and is_scramble_pruned(s1[i:], s2[i:]):
            return True
        if is_scramble_pruned(s1[:i], s2[n - i:]) and is_scramble_pruned(s1[i:], s2[:n - i]):
            return True
    return False


def count_calls_pruned(s1, s2):
    calls = [0]

    def rec(a, b):
        calls[0] += 1
        if a == b:
            return True
        if sorted(a) != sorted(b):
            return False
        n = len(a)
        for i in range(1, n):
            if rec(a[:i], b[:i]) and rec(a[i:], b[i:]):
                return True
            if rec(a[:i], b[n - i:]) and rec(a[i:], b[:n - i]):
                return True
        return False

    return rec(s1, s2), calls[0]


for s1, s2, expected in EXAMPLES + [("abcdefghijk", "bdacefghijk", False)]:
    assert is_scramble_pruned(s1, s2) == expected
    print(f"{s1!r:14} vs {s2!r:14}: Cách 2 = {count_calls_recursive(s1, s2)[1]:6} lời gọi,"
          f" Cách 3 = {count_calls_pruned(s1, s2)[1]:3} lời gọi -> {expected}")

'great'        vs 'rgeat'       : Cách 2 =     12 lời gọi, Cách 3 =   8 lời gọi -> True
'abcde'        vs 'caebd'       : Cách 2 =    109 lời gọi, Cách 3 =   9 lời gọi -> False
'a'            vs 'a'           : Cách 2 =      1 lời gọi, Cách 3 =   1 lời gọi -> True
'abcdefghijk'  vs 'bdacefghijk' : Cách 2 =  82317 lời gọi, Cách 3 = 1023 lời gọi -> False


Khác biệt rất lớn: Ví dụ 2 từ 109 xuống 9 lời gọi; `"abcdefghijk"` / `"bdacefghijk"` từ 82317 xuống 1023 (giảm khoảng 80 lần). Nhưng hãy để ý con số `1023 = 2^10 - 1` — một lũy thừa của 2, dấu hiệu là vẫn còn tăng theo hàm mũ (đoạn đuôi `"efghijk"` giống hệt nhau nên mọi cách cắt trong đó đều "có vẻ được"). Ta sẽ quay lại điểm này ngay sau trace.

Trace dưới cho thấy cắt tỉa hoạt động ở đâu, với cả hai ví dụ của đề.

In [16]:
def trace_pruned(s1, s2, depth=0, tag="gốc"):
    pad = "  " * depth
    if s1 == s2:
        print(f"{pad}[{tag}] {s1!r} vs {s2!r}: giống hệt -> True")
        return True
    if sorted(s1) != sorted(s2):
        print(f"{pad}[{tag}] {s1!r} vs {s2!r}: khác bộ chữ -> False")
        return False
    print(f"{pad}[{tag}] {s1!r} vs {s2!r}: cùng bộ chữ, thử các điểm cắt")
    n = len(s1)
    for i in range(1, n):
        if trace_pruned(s1[:i], s2[:i], depth + 1, f"i={i} giữ") and \
           trace_pruned(s1[i:], s2[i:], depth + 1, f"i={i} giữ"):
            print(f"{pad}  => True (i={i}, không đổi chỗ)")
            return True
        if trace_pruned(s1[:i], s2[n - i:], depth + 1, f"i={i} đổi") and \
           trace_pruned(s1[i:], s2[:n - i], depth + 1, f"i={i} đổi"):
            print(f"{pad}  => True (i={i}, đổi chỗ)")
            return True
    print(f"{pad}  => False")
    return False


trace_pruned("great", "rgeat")
print()
trace_pruned("abcde", "caebd")

[gốc] 'great' vs 'rgeat': cùng bộ chữ, thử các điểm cắt
  [i=1 giữ] 'g' vs 'r': khác bộ chữ -> False
  [i=1 đổi] 'g' vs 't': khác bộ chữ -> False
  [i=2 giữ] 'gr' vs 'rg': cùng bộ chữ, thử các điểm cắt
    [i=1 giữ] 'g' vs 'r': khác bộ chữ -> False
    [i=1 đổi] 'g' vs 'g': giống hệt -> True
    [i=1 đổi] 'r' vs 'r': giống hệt -> True
    => True (i=1, đổi chỗ)
  [i=2 giữ] 'eat' vs 'eat': giống hệt -> True
  => True (i=2, không đổi chỗ)

[gốc] 'abcde' vs 'caebd': cùng bộ chữ, thử các điểm cắt
  [i=1 giữ] 'a' vs 'c': khác bộ chữ -> False
  [i=1 đổi] 'a' vs 'd': khác bộ chữ -> False
  [i=2 giữ] 'ab' vs 'ca': khác bộ chữ -> False
  [i=2 đổi] 'ab' vs 'bd': khác bộ chữ -> False
  [i=3 giữ] 'abc' vs 'cae': khác bộ chữ -> False
  [i=3 đổi] 'abc' vs 'ebd': khác bộ chữ -> False
  [i=4 giữ] 'abcd' vs 'caeb': khác bộ chữ -> False
  [i=4 đổi] 'abcd' vs 'aebd': khác bộ chữ -> False
  => False


False

### Đọc trace Ví dụ 2: vì sao `"caebd"` không phải scramble của `"abcde"`?

`"abcde"` và `"caebd"` cùng bộ chữ nên phải thử cắt. Nhưng **mọi** điểm cắt đều chết ngay ở nửa đầu vì khác bộ chữ:

| `i` | Không đổi chỗ: `s1[:i]` vs `s2[:i]` | Đổi chỗ: `s1[:i]` vs `s2[n-i:]` |
|---|---|---|
| 1 | `"a"` vs `"c"` → khác | `"a"` vs `"d"` → khác |
| 2 | `"ab"` vs `"ca"` → khác | `"ab"` vs `"bd"` → khác |
| 3 | `"abc"` vs `"cae"` → khác | `"abc"` vs `"ebd"` → khác |
| 4 | `"abcd"` vs `"caeb"` → khác | `"abcd"` vs `"aebd"` → khác |

Nói bằng lời: không có cách nào chia `"caebd"` thành "đoạn đầu + đoạn cuối" sao cho một đoạn có đúng bộ chữ của **một tiền tố** `"a"`, `"ab"`, `"abc"`, `"abcd"` của `s1`. Gốc cây không tồn tại → `False`. Tổng cộng chỉ 9 lời gọi.

### Cắt tỉa vẫn vướng: input "đối kháng"

Cắt tỉa chỉ loại được nhánh **khác bộ chữ**. Nếu chuỗi có rất nhiều chữ **lặp lại**, hầu hết các cặp con đều là anagram của nhau, cắt tỉa không giúp gì. Thử họ input:

- `s1 = "abcd" + "x" * k`
- `s2 = "x" * k + "bdac"`

Đáp án luôn là `False` (phần `"abcd"` không thể thành `"bdac"`), nhưng mọi cách cắt khối `x...x` đều "trông có vẻ được".

In [17]:
for k in [4, 8, 12, 16]:
    s1 = "abcd" + "x" * k
    s2 = "x" * k + "bdac"
    ans, calls = count_calls_pruned(s1, s2)
    print(f"n={len(s1):2}  -> {ans}, {calls:7} lời gọi   (2^(n-1) - 1 = {2 ** (len(s1) - 1) - 1})")

n= 8  -> False,     127 lời gọi   (2^(n-1) - 1 = 127)
n=12  -> False,    2047 lời gọi   (2^(n-1) - 1 = 2047)
n=16  -> False,   32767 lời gọi   (2^(n-1) - 1 = 32767)


n=20  -> False,  524287 lời gọi   (2^(n-1) - 1 = 524287)


Số lời gọi đúng bằng `2^(n-1) - 1`: **gấp đôi** mỗi khi thêm 1 ký tự. Với `n = 30` (k = 26) là `2^29 - 1 ≈ 5.4 · 10^8` lời gọi, mỗi lời gọi còn phải `sorted` → nhiều phút trong Python, chắc chắn **Time Limit Exceeded**.

Vậy nguồn lãng phí thứ hai — **lặp lại bài toán con** — có thật không? Ta đếm số cặp `(a, b)` **khác nhau** thực sự được xét, so với tổng số lời gọi:

In [18]:
def count_distinct_pairs(s1, s2):
    calls = [0]
    seen_pairs = set()

    def rec(a, b):
        calls[0] += 1
        seen_pairs.add((a, b))
        if a == b:
            return True
        if sorted(a) != sorted(b):
            return False
        n = len(a)
        for i in range(1, n):
            if rec(a[:i], b[:i]) and rec(a[i:], b[i:]):
                return True
            if rec(a[:i], b[n - i:]) and rec(a[i:], b[:n - i]):
                return True
        return False

    rec(s1, s2)
    return calls[0], len(seen_pairs)


for k in [4, 8, 12, 16]:
    calls, distinct = count_distinct_pairs("abcd" + "x" * k, "x" * k + "bdac")
    print(f"n={4 + k:2}: {calls:7} lời gọi nhưng chỉ {distinct:3} cặp (a, b) khác nhau")

n= 8:     127 lời gọi nhưng chỉ  27 cặp (a, b) khác nhau
n=12:    2047 lời gọi nhưng chỉ  47 cặp (a, b) khác nhau
n=16:   32767 lời gọi nhưng chỉ  67 cặp (a, b) khác nhau


n=20:  524287 lời gọi nhưng chỉ  87 cặp (a, b) khác nhau


Hơn **nửa triệu** lời gọi nhưng chỉ **vài chục** cặp khác nhau! Gần như toàn bộ công sức là giải lại những câu hỏi đã có đáp án. Đây chính là lúc dùng **memoization**.

**Độ phức tạp Cách 3:** trường hợp xấu nhất vẫn **hàm mũ** (ví dụ trên: `2^(n-1)` lời gọi); trên input ngẫu nhiên thường rất nhanh vì cắt tỉa loại hầu hết nhánh. Bộ nhớ `O(n^2)` cho call stack.

## 7. Cách 4 — Đệ quy + memo (top-down DP)

### Kiến thức nền: memoization (ghi nhớ kết quả)

**Memoization** = lưu kết quả của mỗi bài toán con vào một "cuốn sổ" (thường là `dict`). Trước khi giải, tra sổ: có rồi thì trả luôn; chưa có thì giải, ghi vào sổ, rồi trả về. Điều kiện để dùng được: hàm phải là **hàm thuần** — cùng đầu vào luôn cho cùng kết quả.

Ví dụ mini kinh điển: số Fibonacci `fib(n) = fib(n-1) + fib(n-2)`. Bản đệ quy thuần tính `fib(3)` rất nhiều lần.

In [19]:
def fib_plain(n, calls):
    calls[0] += 1
    return n if n < 2 else fib_plain(n - 1, calls) + fib_plain(n - 2, calls)


def fib_memo(n, memo, calls):
    calls[0] += 1
    if n in memo:                 # tra sổ trước
        return memo[n]
    result = n if n < 2 else fib_memo(n - 1, memo, calls) + fib_memo(n - 2, memo, calls)
    memo[n] = result              # ghi vào sổ
    return result


c1, c2 = [0], [0]
print("fib(25) thuần :", fib_plain(25, c1), "| lời gọi:", c1[0])
print("fib(25) memo  :", fib_memo(25, {}, c2), "| lời gọi:", c2[0])

fib(25) thuần : 75025 | lời gọi: 242785
fib(25) memo  : 75025 | lời gọi: 49


### `functools.lru_cache`: memo tự động bằng decorator

Viết `@lru_cache(maxsize=None)` ngay trên định nghĩa hàm, Python sẽ tự làm "cuốn sổ" giúp ta: khóa là bộ tham số, giá trị là kết quả. Tham số phải **hashable** (dùng làm key của dict được): `int`, `str`, `tuple` thì được; `list` thì **không**.

- `f.cache_info()` cho biết số lần trúng sổ (`hits`), số lần phải tính (`misses`), số mục trong sổ (`currsize`).
- `f.cache_clear()` xóa sổ.

In [20]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(80))
print(fib.cache_info())

23416728348467685
CacheInfo(hits=78, misses=81, maxsize=None, currsize=81)


### Áp dụng vào bài: hai cách chọn "khóa" cho cuốn sổ

**Phiên bản A — khóa là cặp chuỗi `(a, b)`**: thêm `@lru_cache` vào hàm của Cách 3. Rất dễ viết. Nhược điểm nhỏ: mỗi lần gọi vẫn phải **cắt chuỗi** (slicing, tốn `O(L)`) và băm (hash) chuỗi làm khóa.

**Phiên bản B — khóa là 3 số `(i1, i2, length)`**: mọi bài toán con đều có dạng "đoạn `s1[i1 : i1+length]` có phải scramble của đoạn `s2[i2 : i2+length]`?". Vậy chỉ cần lưu **vị trí bắt đầu** trong mỗi chuỗi và **độ dài** — không cần tạo chuỗi con mới để làm khóa.

Dịch công thức của insight sang chỉ số, với điểm cắt `k` (`1 ≤ k < length`):

| Trường hợp | Nửa thứ nhất | Nửa thứ hai |
|---|---|---|
| Không đổi chỗ | `f(i1, i2, k)` | `f(i1 + k, i2 + k, length - k)` |
| Đổi chỗ | `f(i1, i2 + length - k, k)` | `f(i1 + k, i2, length - k)` |

Kiểm tra bằng số: `s1 = "great"`, `s2 = "rgeat"`, gốc là `f(0, 0, 5)`, `k = 2`, đổi chỗ → nửa thứ nhất `f(0, 3, 2)` = `"gr"` vs `s2[3:5] = "at"`; nửa thứ hai `f(2, 0, 3)` = `"eat"` vs `s2[0:3] = "rge"`. Đúng với bảng ở mục 4.

In [21]:
def is_scramble_memo_str(s1, s2):
    @lru_cache(maxsize=None)
    def f(a, b):
        if a == b:
            return True
        if sorted(a) != sorted(b):
            return False
        n = len(a)
        for i in range(1, n):
            if f(a[:i], b[:i]) and f(a[i:], b[i:]):
                return True
            if f(a[:i], b[n - i:]) and f(a[i:], b[:n - i]):
                return True
        return False

    result = f(s1, s2)
    return result, f.cache_info()


s1, s2 = "abcd" + "x" * 26, "x" * 26 + "bdac"       # n = 30: Cách 3 cần ~5.4e8 lời gọi
print(len(s1), is_scramble_memo_str(s1, s2))

30 (False, CacheInfo(hits=728, misses=137, maxsize=None, currsize=137))


In [22]:
def is_scramble_memo(s1, s2):
    @lru_cache(maxsize=None)
    def f(i1, i2, length):
        a = s1[i1:i1 + length]
        b = s2[i2:i2 + length]
        if a == b:
            return True
        if sorted(a) != sorted(b):
            return False
        for k in range(1, length):
            # không đổi chỗ
            if f(i1, i2, k) and f(i1 + k, i2 + k, length - k):
                return True
            # đổi chỗ
            if f(i1, i2 + length - k, k) and f(i1 + k, i2, length - k):
                return True
        return False

    return f(0, 0, len(s1))


for s1, s2, expected in EXAMPLES:
    assert is_scramble_memo(s1, s2) == expected
print("3 ví dụ của đề: PASS")

hard_cases = [
    ("abcd" + "x" * 26, "x" * 26 + "bdac"),
    ("eebaacbcbcadaaedceaaacadccd", "eadcaacabaddaceacbceaabeccd"),
]
for s1, s2 in hard_cases:
    print(f"n={len(s1)} -> {is_scramble_memo(s1, s2)}")

3 ví dụ của đề: PASS
n=30 -> False
n=27 -> False


Cặp thứ hai (`n = 27`, chữ cái lặp nhiều) là kiểu input mà Cách 3 không có memo chạy rất lâu (thử nghiệm khi soạn bài: hơn 20 triệu lời gọi mà chưa xong sau 10 giây). Với memo, mỗi cặp `(i1, i2, length)` chỉ được giải **một lần**, nên chạy tức thì.

> Lưu ý: hàm `f` được định nghĩa **bên trong** `is_scramble_memo`, nên mỗi lần gọi `is_scramble_memo` có một cuốn sổ **mới**. Điều này quan trọng: khóa `(i1, i2, length)` chỉ có nghĩa với **đúng cặp** `s1`, `s2` hiện tại (xem Lỗi 5).

### Độ phức tạp Cách 4

Gọi `n = len(s1)`.

- **Số trạng thái:** `length` từ 1 tới `n`; với mỗi `length`, `i1` và `i2` mỗi cái có `n - length + 1` giá trị → tổng `Σ (n-L+1)^2 ≈ n^3 / 3`, tức `O(n^3)`.
- **Mỗi trạng thái:** thử `length - 1` điểm cắt × 2 trường hợp = `O(n)` phép tra sổ; cộng thêm cắt chuỗi + `sorted` `O(n log n)`.
- **Thời gian:** `O(n^3) · O(n) = O(n^4)`. Với `n = 30`: `30^4 = 810 000` — rất nhỏ.
- **Bộ nhớ:** `O(n^3)` cho cuốn sổ + `O(n)` độ sâu call stack.

## 8. Cách 5 — DP 3 chiều bottom-up `dp[length][i][j]`

Memo là "từ trên xuống" (top-down): bắt đầu từ bài lớn, gặp bài con nào thì giải bài đó. **Bottom-up** làm ngược lại: giải **tất cả** bài toán con theo thứ tự từ nhỏ đến lớn, điền vào một bảng, sao cho khi cần một ô thì các ô nó phụ thuộc **đã có sẵn**.

**Định nghĩa ô:**

> `dp[L][i][j] = True` nếu đoạn `s1[i : i+L]` là scramble của đoạn `s2[j : j+L]`.

Đáp án: `dp[n][0][0]`.

**Base case (L = 1):** `dp[1][i][j] = (s1[i] == s2[j])`.

**Công thức chuyển (L ≥ 2):** `dp[L][i][j] = True` nếu tồn tại `k ∈ [1, L-1]` sao cho

- không đổi chỗ: `dp[k][i][j] and dp[L-k][i+k][j+k]`, **hoặc**
- đổi chỗ: `dp[k][i][j+L-k] and dp[L-k][i+k][j]`.

**Thứ tự điền:** mọi ô ở vế phải có độ dài `k` hoặc `L - k`, đều **nhỏ hơn** `L`. Vậy vòng lặp ngoài cùng là `L` tăng dần từ 2 tới `n`; bên trong là `i`, `j` (thứ tự nào cũng được).

### Kiến thức nền: tạo list 3 chiều cho đúng

Cách tự nhiên nhất `[[False] * n] * n` là **sai**: phép `* n` ở ngoài **không sao chép** list bên trong, mà tạo `n` "tên gọi" trỏ tới **cùng một** list (hiện tượng **aliasing**). Sửa một hàng là mọi hàng đổi theo. Cách đúng là dùng **list comprehension** để mỗi hàng là một list riêng.

In [23]:
bad = [[0] * 3] * 2
bad[0][0] = 9
print("sai :", bad, "<- cả 2 hàng đổi!")

good = [[0] * 3 for _ in range(2)]
good[0][0] = 9
print("đúng:", good)

n = 3
dp3 = [[[False] * n for _ in range(n)] for _ in range(n + 1)]   # dp3[L][i][j], L = 0..n
print("kích thước:", len(dp3), "x", len(dp3[0]), "x", len(dp3[0][0]))

sai : [[9, 0, 0], [9, 0, 0]] <- cả 2 hàng đổi!
đúng: [[9, 0, 0], [0, 0, 0]]
kích thước: 4 x 3 x 3


In [24]:
def is_scramble_dp(s1, s2):
    n = len(s1)
    if n != len(s2):
        return False
    # dp[L][i][j]: s1[i:i+L] có phải scramble của s2[j:j+L]?  (L = 0..n; hàng L = 0 bỏ trống)
    dp = [[[False] * n for _ in range(n)] for _ in range(n + 1)]

    for i in range(n):                         # base case L = 1
        for j in range(n):
            dp[1][i][j] = s1[i] == s2[j]

    for L in range(2, n + 1):                  # độ dài tăng dần
        for i in range(n - L + 1):
            for j in range(n - L + 1):
                for k in range(1, L):
                    if (dp[k][i][j] and dp[L - k][i + k][j + k]) or \
                       (dp[k][i][j + L - k] and dp[L - k][i + k][j]):
                        dp[L][i][j] = True
                        break                  # đã tìm thấy một k là đủ
    return dp[n][0][0]


for s1, s2, expected in EXAMPLES:
    assert is_scramble_dp(s1, s2) == expected
for s1, s2 in hard_cases:
    print(f"n={len(s1)} -> {is_scramble_dp(s1, s2)}")
print("3 ví dụ của đề: PASS")

n=30 -> False
n=27 -> False
3 ví dụ của đề: PASS


### Trace Cách 5: in bảng DP từng tầng cho `"great"` / `"rgeat"`

Hàm `trace_dp` chạy đúng thuật toán Cách 5 rồi in từng tầng `L`. Mỗi tầng là một ma trận: **hàng** `i` ứng với đoạn `s1[i:i+L]`, **cột** `j` ứng với đoạn `s2[j:j+L]`; `T` là True, `.` là False. Với mỗi ô True ở tầng `L ≥ 2`, ta còn ghi lại `k` và trường hợp đầu tiên làm nó đúng (in ở cuối).

In [25]:
def trace_dp(s1, s2, explain=()):
    n = len(s1)
    dp = [[[False] * n for _ in range(n)] for _ in range(n + 1)]
    why = {}
    for i in range(n):
        for j in range(n):
            dp[1][i][j] = s1[i] == s2[j]
    for L in range(2, n + 1):
        for i in range(n - L + 1):
            for j in range(n - L + 1):
                for k in range(1, L):
                    if dp[k][i][j] and dp[L - k][i + k][j + k]:
                        dp[L][i][j], why[(L, i, j)] = True, f"k={k} không đổi chỗ"
                        break
                    if dp[k][i][j + L - k] and dp[L - k][i + k][j]:
                        dp[L][i][j], why[(L, i, j)] = True, f"k={k} đổi chỗ"
                        break
    for L in range(1, n + 1):
        m = n - L + 1
        print(f"L={L}:".ljust(n + 3) + " ".join(f"{s2[j:j + L]:>{max(L, 3)}}" for j in range(m)))
        for i in range(m):
            print(f"  {s1[i:i + L]:>{n}} " + " ".join(f"{'T' if dp[L][i][j] else '.':>{max(L, 3)}}" for j in range(m)))
    for key in explain:
        L, i, j = key
        print(f"dp[{L}][{i}][{j}] ({s1[i:i + L]!r} vs {s2[j:j + L]!r}) = {dp[L][i][j]}: {why.get(key, '-')}")
    return dp[n][0][0]


trace_dp("great", "rgeat", explain=[(2, 0, 0), (3, 2, 2), (5, 0, 0)])

L=1:      r   g   e   a   t
      g   .   T   .   .   .
      r   T   .   .   .   .
      e   .   .   T   .   .
      a   .   .   .   T   .
      t   .   .   .   .   T
L=2:     rg  ge  ea  at
     gr   T   .   .   .
     re   .   .   .   .
     ea   .   .   T   .
     at   .   .   .   T
L=3:    rge gea eat
    gre   T   .   .
    rea   .   .   .
    eat   .   .   T
L=4:    rgea geat
   grea    T    .
   reat    .    .
L=5:    rgeat
  great     T
dp[2][0][0] ('gr' vs 'rg') = True: k=1 đổi chỗ
dp[3][2][2] ('eat' vs 'eat') = True: k=1 không đổi chỗ
dp[5][0][0] ('great' vs 'rgeat') = True: k=2 không đổi chỗ


True

### Đọc bảng

- **Tầng `L=1`** chỉ là so sánh từng cặp chữ cái: hàng `'g'` có `T` ở cột `'g'` (vị trí 1 của `"rgeat"`), hàng `'r'` có `T` ở cột `'r'` (vị trí 0)...
- **`dp[2][0][0]`** (`"gr"` vs `"rg"`) = T với `k=1` **đổi chỗ**: cần `dp[1][0][1]` (`'g'` vs `'g'`) và `dp[1][1][0]` (`'r'` vs `'r'`) — cả hai là T ở tầng 1.
- **`dp[3][2][2]`** (`"eat"` vs `"eat"`) = T với `k=1` không đổi chỗ: `'e'`~`'e'` và `"at"`~`"at"`.
- **`dp[5][0][0]`** (đáp án) = T với `k=2` **không đổi chỗ**: cần `dp[2][0][0]` (`"gr"`~`"rg"`) và `dp[3][2][2]` (`"eat"`~`"eat"`) — chính là hai ô vừa giải thích. Đây đúng là cây ở mục 2.2!

Khác với memo, bảng bottom-up điền **mọi** ô, kể cả những ô không bao giờ cần tới (ví dụ `"ea"` vs `"rg"`).

### Độ phức tạp Cách 5

- **Thời gian:** 4 vòng lồng nhau `L, i, j, k`, mỗi vòng `≤ n` → `O(n^4)`. Chính xác hơn: `Σ_L (n-L+1)^2 · (L-1) ≈ n^4 / 12`.
- **Bộ nhớ:** bảng `(n+1) × n × n` → `O(n^3)`. Với `n = 30`: khoảng 28 000 ô.

So với Cách 4: cùng `O(n^4)` lý thuyết; Cách 5 không có đệ quy (không lo call stack) và dễ phân tích, còn Cách 4 nhờ cắt tỉa chỉ đi vào những trạng thái **thực sự cần** nên trong Python thường nhanh hơn.

## 9. Code nộp LeetCode

Chọn **Cách 4 (memo theo chỉ số + cắt tỉa)** làm bản chính: ngắn, dễ đọc, chỉ thăm trạng thái cần thiết. Vài điểm cần chú ý khi chuyển sang `class Solution`:

- Hàm đệ quy `f` đặt **bên trong** `isScramble` để dùng được `s1`, `s2` và có cuốn sổ riêng cho mỗi test.
- Thay `sorted(a) != sorted(b)` bằng `Counter` vẫn được; với độ dài ≤ 30 thì `sorted` đủ nhanh và gọn.
- Không cần kiểm tra `len(s1) != len(s2)` vì constraints đảm bảo bằng nhau.

In [26]:
from functools import lru_cache

class Solution:
    def isScramble(self, s1: str, s2: str) -> bool:
        @lru_cache(maxsize=None)
        def f(i1: int, i2: int, length: int) -> bool:
            a = s1[i1:i1 + length]
            b = s2[i2:i2 + length]
            if a == b:
                return True
            if sorted(a) != sorted(b):
                return False
            for k in range(1, length):
                if f(i1, i2, k) and f(i1 + k, i2 + k, length - k):
                    return True
                if f(i1, i2 + length - k, k) and f(i1 + k, i2, length - k):
                    return True
            return False

        return f(0, 0, len(s1))

In [27]:
sol = Solution()

# 3 ví dụ của đề
assert sol.isScramble("great", "rgeat") is True
assert sol.isScramble("abcde", "caebd") is False
assert sol.isScramble("a", "a") is True

# edge case
assert sol.isScramble("a", "b") is False                  # 1 ký tự khác nhau
assert sol.isScramble("ab", "ba") is True                 # đổi chỗ đơn giản nhất
assert sol.isScramble("abc", "abd") is False              # khác bộ chữ
assert sol.isScramble("abcd", "bdac") is False            # cùng bộ chữ nhưng KHÔNG phải scramble
assert sol.isScramble("abcd", "cadb") is False
assert sol.isScramble("abcdefghij", "jihgfedcba") is True # đảo ngược toàn bộ luôn đạt được
assert sol.isScramble("aabb", "abab") is True             # chữ lặp
for s1, s2 in hard_cases:                                 # input đối kháng, n = 30 và 27
    assert sol.isScramble(s1, s2) is False
assert sol.isScramble("a" * 30, "a" * 30) is True         # độ dài tối đa
print("Tất cả test của Solution: PASS")

Tất cả test của Solution: PASS


### Bản nộp thay thế: DP bottom-up

Nếu muốn tránh đệ quy hoàn toàn, có thể nộp Cách 5. Ở đây đặt tên `SolutionDP` để không ghi đè bản chính — **khi nộp lên LeetCode, đổi tên lại thành `Solution`**.

In [28]:
class SolutionDP:
    def isScramble(self, s1: str, s2: str) -> bool:
        n = len(s1)
        dp = [[[False] * n for _ in range(n)] for _ in range(n + 1)]
        for i in range(n):
            for j in range(n):
                dp[1][i][j] = s1[i] == s2[j]
        for L in range(2, n + 1):
            for i in range(n - L + 1):
                for j in range(n - L + 1):
                    for k in range(1, L):
                        if (dp[k][i][j] and dp[L - k][i + k][j + k]) or \
                           (dp[k][i][j + L - k] and dp[L - k][i + k][j]):
                            dp[L][i][j] = True
                            break
        return dp[n][0][0]


for s1, s2, expected in EXAMPLES + [("abcd", "bdac", False), ("aabb", "abab", True)]:
    assert SolutionDP().isScramble(s1, s2) == expected
print("SolutionDP: PASS")

SolutionDP: PASS


## 10. Test đối chiếu chéo trên input ngẫu nhiên

Ý tưởng: dùng **Cách 1 (sinh tất cả)** làm "trọng tài" cho chuỗi ngắn (`n ≤ 6`) — nó chậm nhưng bám sát định nghĩa nên đáng tin. Sau đó so mọi cách còn lại với nó. Hai chuỗi hoàn toàn ngẫu nhiên gần như luôn cho `False` (khác bộ chữ), nên test kiểu đó rất "dễ". Vì vậy hàm `make_pair` trộn 3 loại cặp:

- khoảng 40% tạo bằng `random_scramble` → chắc chắn `True`;
- khoảng 40% là **hoán vị ngẫu nhiên** của `s1` (cùng bộ chữ — loại khó nhất, có thể `True` hoặc `False`);
- khoảng 20% là chuỗi ngẫu nhiên bất kỳ (thường `False` ngay ở bước cắt tỉa).

Bảng chữ cái nhỏ `"abcd"` để có nhiều chữ lặp.

Với chuỗi dài hơn (`n` tới 12), Cách 1 và Cách 2 quá chậm, nên chỉ so Cách 3, 4, 5 và `Solution` với nhau.

In [29]:
rng = random.Random(2024)
stats = Counter()

def make_pair(n, rng):
    s1 = "".join(rng.choice("abcd") for _ in range(n))
    r = rng.random()
    if r < 0.4:                                    # chắc chắn True
        s2 = random_scramble(s1, rng)
    elif r < 0.8:                                  # cùng bộ chữ: loại khó nhất
        letters = list(s1)
        rng.shuffle(letters)
        s2 = "".join(letters)
    else:                                          # chuỗi ngẫu nhiên bất kỳ
        s2 = "".join(rng.choice("abcd") for _ in range(n))
    return s1, s2

for _ in range(400):                       # n nhỏ: có trọng tài Cách 1
    s1, s2 = make_pair(rng.randint(1, 6), rng)
    truth = is_scramble_generate(s1, s2)
    results = [is_scramble_recursive(s1, s2), is_scramble_pruned(s1, s2),
               is_scramble_memo(s1, s2), is_scramble_dp(s1, s2), Solution().isScramble(s1, s2)]
    assert all(r == truth for r in results), (s1, s2, truth, results)
    stats[truth] += 1

for _ in range(300):                       # n lớn hơn: so các cách nhanh với nhau
    s1, s2 = make_pair(rng.randint(7, 12), rng)
    a = is_scramble_memo(s1, s2)
    assert a == is_scramble_dp(s1, s2) == SolutionDP().isScramble(s1, s2) == Solution().isScramble(s1, s2)
    stats[a] += 1

print("Đối chiếu chéo 700 cặp: PASS | số cặp True/False:", stats[True], "/", stats[False])

Đối chiếu chéo 700 cặp: PASS | số cặp True/False: 535 / 165


## 11. Vì sao thuật toán đúng?

Ta chứng minh bằng **quy nạp theo độ dài** `L` của hai đoạn đang so (cũng chính là thứ tự điền bảng ở Cách 5).

**Mệnh đề:** `f(a, b)` (hay `dp[L][i][j]`) trả về `True` **khi và chỉ khi** tồn tại một cây chia đôi (như mục 2.2) biến `a` thành `b`.

- **L = 1:** cây chỉ có một lá, nên `b` phải bằng `a`. Hàm trả về `a == b`. Đúng.
- **L ≥ 2, chiều "nếu có cây thì hàm trả True":** gốc cây cắt `a` tại một `k` nào đó. Nếu gốc không đổi chỗ thì `b[:k]` là kết quả của cây con trái (dài `k`) và `b[k:]` của cây con phải → hai bài toán con độ dài `< L` đều có cây, theo giả thiết quy nạp hàm trả `True` cho cả hai → nhánh "không đổi chỗ" tại `k` đúng. Nếu gốc đổi chỗ, tương tự với `b[L-k:]` và `b[:L-k]`. Vì hàm thử **mọi** `k`, nó không bỏ sót.
- **Chiều "nếu hàm trả True thì có cây":** hàm trả `True` vì một `k` và một trường hợp nào đó đúng; theo quy nạp hai cặp con đều có cây; ghép chúng làm con trái/phải của một nút gốc cắt tại `k` (với `swap` tương ứng) ta được cây cho `(a, b)`. Không bao giờ trả nhầm.

**Các cắt tỉa không làm sai kết quả:** `a == b` → luôn có cây "cắt bất kỳ, không bao giờ đổi chỗ"; khác bộ chữ cái → không cây nào tạo được vì mọi phép biến đổi chỉ sắp xếp lại chữ. **Memo** không đổi kết quả vì `f` là hàm thuần: cùng `(i1, i2, length)` với cùng `s1`, `s2` luôn cho cùng đáp án.

**Invariant bằng một câu:** *"Ô `(L, i, j)` đúng ⇔ đoạn dài `L` bắt đầu tại `i` của `s1` có thể xáo thành đoạn dài `L` bắt đầu tại `j` của `s2`; và mọi ô ngắn hơn đã đúng trước khi ta tính ô này."*

## 12. So sánh các cách

Gọi `n = len(s1)` (≤ 30).

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Dùng khi |
|---|---|---|---|---|
| 1. Sinh tất cả | Liệt kê mọi scramble của `s1` vào `set` | hàm mũ, theo số Schröder (~`5.8^n`) | hàm mũ | Hiểu định nghĩa, trọng tài cho `n ≤ 6` |
| 2. Đệ quy thuần | Thử mọi `i`, 2 trường hợp | hàm mũ (đo được ~`3^n` trên ca `False`) | `O(n^2)` stack | Hiểu insight; không nộp |
| 3. Đệ quy + cắt tỉa | Thêm `a == b`, `sorted` | hàm mũ trường hợp xấu (`2^(n-1)` trên ca đối kháng) | `O(n^2)` stack | Thường nhanh trên input ngẫu nhiên, nhưng có thể TLE |
| 4. Đệ quy + memo | Sổ `(i1, i2, length)` | `O(n^4)` | `O(n^3)` | **Bản nộp chính** |
| 5. DP bottom-up | Bảng `dp[L][i][j]`, `L` tăng dần | `O(n^4)` (≈ `n^4/12`) | `O(n^3)` | Không muốn đệ quy; dễ in bảng để debug |

Mạch tiến hóa: **sinh hết** → (chỉ kiểm tra theo nhát cắt đầu tiên) → **đệ quy** → (bỏ nhánh khác bộ chữ) → **cắt tỉa** → (không giải lại bài con đã giải) → **memo** → (đổi chiều, điền theo độ dài) → **bottom-up**.

## 13. Các lỗi người mới hay gặp

### Lỗi 1: Ghép sai cặp trong trường hợp đổi chỗ

Khi đổi chỗ, `s1[:i]` (dài `i`) phải so với **`i` ký tự cuối** `s2[n-i:]`, và `s1[i:]` (dài `n-i`) so với `s2[:n-i]`. Lỗi rất hay gặp là "quên đổi độ dài", viết `s1[i:]` với `s2[:i]` — chỉ đúng khi tình cờ `i = n - i`. Cell dưới tự tìm phản ví dụ nhỏ nhất bằng cách so với Cách 2 đúng.

In [30]:
def is_scramble_wrong_swap(s1, s2):
    n = len(s1)
    if n == 1:
        return s1 == s2
    if len(s2) != n:                  # bị ghép sai độ dài thì chắc chắn trả False
        return False
    for i in range(1, n):
        if is_scramble_wrong_swap(s1[:i], s2[:i]) and is_scramble_wrong_swap(s1[i:], s2[i:]):
            return True
        # SAI: s1[i:] dài n-i nhưng lại so với s2[:i] dài i
        if is_scramble_wrong_swap(s1[:i], s2[n - i:]) and is_scramble_wrong_swap(s1[i:], s2[:i]):
            return True
    return False


found = None
for n in range(1, 6):
    for s2 in sorted(all_scrambles("abcde"[:n])):
        s1 = "abcde"[:n]
        if is_scramble_wrong_swap(s1, s2) != is_scramble_recursive(s1, s2):
            found = (s1, s2)
            break
    if found:
        break
print("Phản ví dụ nhỏ nhất:", found, "| đúng:", is_scramble_recursive(*found),
      "| bản sai:", is_scramble_wrong_swap(*found))

Phản ví dụ nhỏ nhất: ('abc', 'bca') | đúng: True | bản sai: False


**Sửa:** luôn tự hỏi "nửa này dài bao nhiêu?" — nửa dài `i` đi với đoạn dài `i`. Trong bản chỉ số: `f(i1, i2 + length - k, k)` và `f(i1 + k, i2, length - k)`.

### Lỗi 2: Cho điểm cắt chạy từ `0` hoặc tới `n`

```python
for i in range(0, n):      # SAI: i = 0 tạo phần rỗng
    if rec(s1[:i], s2[:i]) and rec(s1[i:], s2[i:]):   # rec(s1, s2) gọi lại chính nó!
```

Với `i = 0`, nửa phải là `s1[0:]` = **toàn bộ** `s1` → hàm gọi lại đúng bài toán ban đầu → **đệ quy vô hạn** (`RecursionError`). Đề yêu cầu hai phần **khác rỗng**, nên `i` chỉ chạy `range(1, n)`.

### Lỗi 3: Tưởng "cùng bộ chữ cái" là đủ

```python
return sorted(s1) == sorted(s2)     # SAI
```

Phản ví dụ: `"abcd"` và `"bdac"` cùng bộ chữ nhưng không phải scramble (mục 3), và Ví dụ 2 của đề `"abcde"` / `"caebd"` cũng vậy. Kiểm tra anagram chỉ là **cắt tỉa** (điều kiện cần).

### Lỗi 4: Tạo bảng DP bằng phép nhân list, hoặc sai thứ tự vòng lặp

- `[[[False] * n] * n] * (n + 1)` → aliasing, sửa một ô là sửa nhiều ô (mục 8). Dùng list comprehension.
- Đặt vòng `i` hoặc `j` ra ngoài `L` → khi tính `dp[L][i][j]` có thể cần `dp[L-k][i+k][...]` **chưa được tính** (vẫn là `False` mặc định) → trả `False` sai. **`L` phải là vòng ngoài cùng, tăng dần.**

### Lỗi 5: Đặt cache "toàn cục" với khóa chỉ số

Nếu hàm memo theo `(i1, i2, length)` được đặt **ngoài** (ví dụ một hàm toàn cục đọc `s1`, `s2` từ biến toàn cục, hoặc một method có `@lru_cache` dùng chung qua nhiều test), thì cuốn sổ từ test trước bị **dùng lại** cho test sau. Nhưng khóa `(0, 0, 2)` của test `("ab", "ba")` và của test `("ab", "cd")` là **hai câu hỏi khác nhau**!

In [31]:
S1 = S2 = ""

@lru_cache(maxsize=None)
def f_global(i1, i2, length):          # SAI: khóa không chứa S1, S2
    a, b = S1[i1:i1 + length], S2[i2:i2 + length]
    if a == b:
        return True
    if sorted(a) != sorted(b):
        return False
    return any((f_global(i1, i2, k) and f_global(i1 + k, i2 + k, length - k)) or
               (f_global(i1, i2 + length - k, k) and f_global(i1 + k, i2, length - k))
               for k in range(1, length))

def is_scramble_global_cache(s1, s2):
    global S1, S2
    S1, S2 = s1, s2
    return f_global(0, 0, len(s1))

print("test 1 ('ab','ba'):", is_scramble_global_cache("ab", "ba"))   # True, đúng
print("test 2 ('ab','cd'):", is_scramble_global_cache("ab", "cd"), "<- SAI, phải là False (lấy từ sổ cũ)")
print("bản đúng           :", Solution().isScramble("ab", "cd"))

test 1 ('ab','ba'): True
test 2 ('ab','cd'): True <- SAI, phải là False (lấy từ sổ cũ)
bản đúng           : False


**Sửa:** định nghĩa hàm memo **bên trong** `isScramble` (như bản `Solution`), hoặc gọi `f.cache_clear()` mỗi lần, hoặc dùng khóa là chính các chuỗi con `(a, b)`.

### Lỗi 6: Quên `return True` ngay khi tìm thấy, hoặc `return False` quá sớm

```python
for i in range(1, n):
    return (rec(...) and rec(...)) or (rec(...) and rec(...))   # SAI: chỉ thử i = 1 rồi thoát
```

`return` nằm **trong** vòng lặp nên chỉ thử `i = 1`. Đúng là: gặp `True` thì trả ngay, còn `return False` phải đặt **sau** vòng lặp, khi đã thử hết mọi `i`.

## 14. Mẹo nhận dạng pattern

Khi đề có các dấu hiệu sau, hãy nghĩ tới **"thử mọi điểm cắt + đệ quy hai nửa" → memo / interval DP**:

- Định nghĩa bài toán **đệ quy theo cách chia đôi** một chuỗi/mảng ("chia thành hai phần khác rỗng, rồi làm tiếp trên từng phần").
- Câu hỏi dạng **có/không tồn tại**, **đếm số cách**, hoặc **min/max** trên tất cả cách chia.
- Ràng buộc nhỏ (`n ≤ 30`, `n ≤ 100`, `n ≤ 500`) — gợi ý độ phức tạp `O(n^3)` hoặc `O(n^4)`.

Quy trình tư duy dùng lại được:

1. Viết đệ quy thuần **bám sát định nghĩa** (Cách 2).
2. Tìm **trạng thái tối thiểu** mô tả một bài toán con: ở đây là `(i1, i2, length)` — 2 vị trí bắt đầu + 1 độ dài (vì hai đoạn luôn **cùng độ dài**, không cần 4 chỉ số).
3. Thêm memo → top-down DP; muốn bottom-up thì xác định **thứ tự phụ thuộc** (ở đây: độ dài tăng dần).
4. Tìm **điều kiện cần rẻ** để cắt tỉa (bộ chữ cái, tổng, độ dài...).

## 15. Biến thể và bài liên quan

- **Đếm số cây xáo trộn** biến `s1` thành `s2`: thay `or` bằng `+`, `and` bằng `*` trong công thức (Bài 6 bên dưới).
- **Trả về một kịch bản cụ thể** (cây chia đôi) chứ không chỉ `True/False` — truy vết ngược điểm cắt `k` (Bài 5).
- **LeetCode 242 — Valid Anagram (Easy):** chính là phép cắt tỉa của bài này.
- **LeetCode 97 — Interleaving String (Medium):** DP trên hai chuỗi, cũng có thể làm đệ quy + memo rồi bottom-up.
- **LeetCode 241 — Different Ways to Add Parentheses (Medium):** thử mọi điểm cắt (mọi toán tử), đệ quy hai nửa, ghép kết quả.
- **LeetCode 312 — Burst Balloons (Hard), 1039 — Minimum Score Triangulation of Polygon (Medium), 664 — Strange Printer (Hard):** interval DP — trạng thái là một đoạn `[i, j]`, thử mọi điểm chia `k` bên trong.
- **LeetCode 10 — Regular Expression Matching, 44 — Wildcard Matching, 72 — Edit Distance:** cùng lộ trình đệ quy → memo → bảng DP, nhưng trạng thái 2 chiều.
- Góc toán: với chữ cái đôi một khác nhau, các scramble của `s1` chính là các **hoán vị tách được (separable permutations)**; số lượng là số Schröder lớn `1, 2, 6, 22, 90, 394, ...` như ta đã đếm ở mục 3.

## 16. Bài tập vận dụng

Làm theo thứ tự, tự làm trước rồi mới xem **Gợi ý** (mục 17) và **Đáp án** (mục 18). Các cell TODO chỉ định nghĩa hàm, **chưa được gọi**; test để sẵn ở dạng comment — làm xong thì bỏ dấu `#` để tự kiểm tra.

### Bài 1 — Đoán output (không chạy code)

Với mỗi cặp, `s2` có phải scramble của `s1` không? Giải thích ngắn bằng "nhát cắt đầu tiên".

1. `("ab", "ba")`
2. `("abc", "bca")`
3. `("abc", "abd")`
4. `("abcd", "cdab")`
5. `("abcde", "edcba")`
6. `("abcdx", "bdacx")`

### Bài 2 — Tính tay bảng DP

Với `s1 = "abc"`, `s2 = "bca"`, hãy tự điền bảng `dp[L][i][j]` cho `L = 1` (3×3), `L = 2` (2×2), `L = 3` (1×1). Ở ô `dp[3][0][0]`, ghi rõ `k` nào và trường hợp nào (giữ / đổi chỗ) làm nó đúng.

### Bài 3 — Điền TODO: đệ quy + memo theo chỉ số

Hoàn thiện hàm dưới (không nhìn lại mục 7 nếu có thể). Trạng thái `f(i1, i2, length)`.

In [32]:
def exercise_scramble_memo(s1: str, s2: str) -> bool:
    @lru_cache(maxsize=None)
    def f(i1, i2, length):
        a = s1[i1:i1 + length]
        b = s2[i2:i2 + length]
        # TODO 1: nếu a == b thì trả True
        # TODO 2: nếu a, b khác bộ chữ cái thì trả False
        # TODO 3: thử mọi k từ 1 tới length - 1
        #         - không đổi chỗ: f(i1, i2, k) and f(?, ?, ?)
        #         - đổi chỗ     : f(i1, ?, k) and f(?, i2, ?)
        # TODO 4: thử hết mà không được thì trả False
        pass

    return f(0, 0, len(s1))


# Làm xong rồi mới mở test:
# assert exercise_scramble_memo("great", "rgeat") is True
# assert exercise_scramble_memo("abcde", "caebd") is False
# assert exercise_scramble_memo("a", "a") is True
# assert exercise_scramble_memo("abcd", "bdac") is False

### Bài 4 — Tự viết lại DP bottom-up

Viết lại Cách 5 từ đầu: tạo bảng 3 chiều **đúng cách**, điền base case `L = 1`, rồi `L` tăng dần. Thử viết mà không nhìn code ở mục 8.

In [33]:
def exercise_scramble_dp(s1: str, s2: str) -> bool:
    # TODO: dp[L][i][j] = s1[i:i+L] có phải scramble của s2[j:j+L]?
    raise NotImplementedError


# assert exercise_scramble_dp("great", "rgeat") is True
# assert exercise_scramble_dp("abcde", "caebd") is False
# assert exercise_scramble_dp("aabb", "abab") is True
# assert exercise_scramble_dp("abcdx", "bdacx") is False

### Bài 5 — Biến thể: trả về **kịch bản** (cây chia đôi)

Thay vì `True/False`, hãy trả về một cây theo đúng định dạng ở mục 2.2 — lá là chuỗi, nút là `(trái, phải, swap)` — sao cho `apply_tree(tree) == s2` và `tree_source(tree) == s1`. Nếu không tồn tại, trả về `None`. Khi `a == b`, được phép trả luôn lá `a` (nghĩa là "không đổi gì").

In [34]:
def exercise_scramble_tree(s1: str, s2: str):
    # TODO: giống Cách 4 nhưng f trả về cây (hoặc None) thay vì bool.
    #       - a == b              -> trả lá a
    #       - khác bộ chữ cái     -> None
    #       - không đổi chỗ tại k : (cây(s1[:k], s2[:k]), cây(s1[k:], s2[k:]), False)
    #       - đổi chỗ tại k       : (cây(s1[:k], s2[n-k:]), cây(s1[k:], s2[:n-k]), True)
    raise NotImplementedError


# t = exercise_scramble_tree("great", "rgeat")
# print_tree(t)
# assert apply_tree(t) == "rgeat" and tree_source(t) == "great"
# assert exercise_scramble_tree("abcde", "caebd") is None

### Bài 6 — Mini challenge: đếm số cây xáo trộn

Đếm số **cây chia đôi khác nhau** (mỗi nút: một điểm cắt + một quyết định đổi/không đổi; mọi lá dài đúng 1) biến `s1` thành `s2`. Hai cây khác nhau nếu khác điểm cắt hoặc khác quyết định ở ít nhất một nút.

Ví dụ: `("ab", "ba")` → 1 cây; `("aa", "aa")` → 2 cây (cắt `"a"|"a"`, đổi hoặc không đều ra `"aa"`); `("abc", "abc")` → 2 cây.

Gợi ý kiểm tra: với `s1 = s2 = "abcd"` (chữ khác nhau) đáp án là số Catalan `C_3 = 5`. Ở bài này **không** được dùng cắt tỉa `a == b → 1` (vì một cặp giống nhau vẫn có thể có nhiều cây)!

In [35]:
def exercise_count_trees(s1: str, s2: str) -> int:
    # TODO: memo theo (i1, i2, length); base case length == 1 -> 1 hoặc 0
    #       tổng qua k: count(không đổi) + count(đổi chỗ), với count(cặp) = tích số cây hai nửa
    raise NotImplementedError


# assert exercise_count_trees("ab", "ba") == 1
# assert exercise_count_trees("aa", "aa") == 2
# assert exercise_count_trees("abc", "abc") == 2
# assert exercise_count_trees("abcd", "abcd") == 5
# assert exercise_count_trees("abcde", "caebd") == 0

## 17. Gợi ý theo mức

**Bài 1**
- Mức 1: chữ cái có khớp không? Nếu khớp, thử từng nhát cắt đầu tiên của `s1`.
- Mức 2: đảo ngược toàn bộ luôn đạt được (cắt bất kỳ, **luôn** đổi chỗ, làm đệ quy). `"cdab"` = `"cd" + "ab"`.
- Mức 3: câu 6 — chữ `x` ở cuối cả hai chuỗi; thử cắt `"abcd" | "x"`: không đổi chỗ thì cần `"abcd"` ~ `"bdac"` (đã biết là False). Các nhát cắt khác có cứu được không?

**Bài 2**
- Mức 1: tầng `L = 1` chỉ là so sánh chữ cái `s1[i] == s2[j]`.
- Mức 2: `dp[2][0][j]` là `"ab"` vs `s2[j:j+2]`; `dp[2][1][j]` là `"bc"` vs `s2[j:j+2]`.
- Mức 3: `dp[3][0][0]`: thử `k = 1` đổi chỗ — cần `dp[1][0][2]` (`'a'` vs `'a'`) và `dp[2][1][0]` (`"bc"` vs `"bc"`).

**Bài 3**
- Mức 1: chép bảng "slicing → chỉ số" ở mục 7.
- Mức 2: không đổi chỗ: `f(i1 + k, i2 + k, length - k)`.
- Mức 3: đổi chỗ: `f(i1, i2 + length - k, k) and f(i1 + k, i2, length - k)`.

**Bài 4**
- Mức 1: tạo bảng bằng 3 tầng list comprehension, kích thước `(n+1) × n × n`.
- Mức 2: `L` là vòng ngoài cùng, chạy `range(2, n + 1)`; `i`, `j` chạy `range(n - L + 1)`.
- Mức 3: ô đúng khi `(dp[k][i][j] and dp[L-k][i+k][j+k]) or (dp[k][i][j+L-k] and dp[L-k][i+k][j])`.

**Bài 5**
- Mức 1: chỗ nào Cách 4 trả `True`, giờ trả một cây; chỗ trả `False` thì trả `None`.
- Mức 2: kiểm tra `left is not None` trước khi gọi nửa còn lại (giữ short-circuit).
- Mức 3: đặt memo theo `(i1, i2, length)`; trả về `(left, right, False)` hoặc `(left, right, True)`.

**Bài 6**
- Mức 1: đây là "đổi phép toán" trên cùng một khung: `or → +`, `and → *`, `True → 1`, `False → 0`.
- Mức 2: base case `length == 1`: trả `1 if s1[i1] == s2[i2] else 0`. Có thể giữ cắt tỉa "khác bộ chữ → 0" vì nó vẫn đúng.
- Mức 3: `total += f(i1, i2, k) * f(i1+k, i2+k, length-k) + f(i1, i2+length-k, k) * f(i1+k, i2, length-k)`.

## 18. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

1. `True` — cắt `"a"|"b"`, đổi chỗ.
2. `True` — cắt `"a"|"bc"`, đổi chỗ: `"bc" + "a"`.
3. `False` — khác bộ chữ cái (`'c'` so với `'d'`).
4. `True` — cắt `"ab"|"cd"`, đổi chỗ: `"cd" + "ab"`.
5. `True` — đảo ngược toàn bộ: mọi nút đều đổi chỗ.
6. `False` — nhát cắt đầu tiên của `"abcdx"` chỉ có thể là `"a"|"bcdx"`, `"ab"|"cdx"`, `"abc"|"dx"`, `"abcd"|"x"`. Đoạn đầu hoặc đoạn cuối của `"bdacx"` có cùng bộ chữ với một trong các khối đó chỉ khi khối là `"abcd"` (đoạn đầu `"bdac"`) — nhưng `"abcd"` ~ `"bdac"` là False. Các khối `{a}`, `{a,b}`, `{a,b,c}` không xuất hiện thành đoạn liền ở đầu hay cuối `"bdacx"`.

### Đáp án Bài 2

Cell dưới kiểm chứng Bài 1 và in bảng của Bài 2 bằng `trace_dp`.

In [36]:
quiz = [("ab", "ba"), ("abc", "bca"), ("abc", "abd"), ("abcd", "cdab"), ("abcde", "edcba"), ("abcdx", "bdacx")]
answers = [True, True, False, True, True, False]
for (s1, s2), ans in zip(quiz, answers):
    assert Solution().isScramble(s1, s2) == ans == is_scramble_generate(s1, s2)
    print(f"{s1!r:8} vs {s2!r:8} -> {ans}")

print("\nBài 2 — bảng DP cho s1='abc', s2='bca':")
trace_dp("abc", "bca", explain=[(2, 1, 0), (3, 0, 0)])

'ab'     vs 'ba'     -> True
'abc'    vs 'bca'    -> True
'abc'    vs 'abd'    -> False
'abcd'   vs 'cdab'   -> True
'abcde'  vs 'edcba'  -> True
'abcdx'  vs 'bdacx'  -> False

Bài 2 — bảng DP cho s1='abc', s2='bca':
L=1:    b   c   a
    a   .   .   T
    b   T   .   .
    c   .   T   .
L=2:   bc  ca
   ab   .   .
   bc   T   .
L=3:  bca
  abc   T
dp[2][1][0] ('bc' vs 'bc') = True: k=1 không đổi chỗ
dp[3][0][0] ('abc' vs 'bca') = True: k=1 đổi chỗ


True

Đọc bảng Bài 2: tầng 1 có `T` ở `('a','a')`, `('b','b')`, `('c','c')`. Tầng 2: `"ab"` không khớp `"bc"` hay `"ca"` (khác bộ chữ); `"bc"` khớp `"bc"` (giống hệt, `k=1` không đổi chỗ). Tầng 3: `dp[3][0][0]` = T với **`k=1` đổi chỗ** — `'a'` ~ `s2[2] = 'a'` và `"bc"` ~ `s2[0:2] = "bc"`.

In [37]:
# ===== Đáp án Bài 3 và Bài 4 =====
def answer_scramble_memo(s1: str, s2: str) -> bool:
    @lru_cache(maxsize=None)
    def f(i1, i2, length):
        a = s1[i1:i1 + length]
        b = s2[i2:i2 + length]
        if a == b:
            return True
        if sorted(a) != sorted(b):
            return False
        for k in range(1, length):
            if f(i1, i2, k) and f(i1 + k, i2 + k, length - k):
                return True
            if f(i1, i2 + length - k, k) and f(i1 + k, i2, length - k):
                return True
        return False

    return f(0, 0, len(s1))


def answer_scramble_dp(s1: str, s2: str) -> bool:
    n = len(s1)
    dp = [[[False] * n for _ in range(n)] for _ in range(n + 1)]
    for i in range(n):
        for j in range(n):
            dp[1][i][j] = s1[i] == s2[j]
    for L in range(2, n + 1):
        for i in range(n - L + 1):
            for j in range(n - L + 1):
                dp[L][i][j] = any(
                    (dp[k][i][j] and dp[L - k][i + k][j + k]) or
                    (dp[k][i][j + L - k] and dp[L - k][i + k][j])
                    for k in range(1, L)
                )
    return dp[n][0][0]


for fn in (answer_scramble_memo, answer_scramble_dp):
    assert fn("great", "rgeat") is True
    assert fn("abcde", "caebd") is False
    assert fn("a", "a") is True
    assert fn("abcd", "bdac") is False
    assert fn("aabb", "abab") is True
    assert fn("abcdx", "bdacx") is False
print("Bài 3, Bài 4: PASS")

Bài 3, Bài 4: PASS


In [38]:
# ===== Đáp án Bài 5: trả về cây chia đôi =====
def answer_scramble_tree(s1: str, s2: str):
    @lru_cache(maxsize=None)
    def f(i1, i2, length):
        a = s1[i1:i1 + length]
        b = s2[i2:i2 + length]
        if a == b:
            return a                                   # lá: không đổi gì
        if sorted(a) != sorted(b):
            return None
        for k in range(1, length):
            left = f(i1, i2, k)
            if left is not None:
                right = f(i1 + k, i2 + k, length - k)
                if right is not None:
                    return (left, right, False)        # không đổi chỗ
            left = f(i1, i2 + length - k, k)
            if left is not None:
                right = f(i1 + k, i2, length - k)
                if right is not None:
                    return (left, right, True)         # đổi chỗ
        return None

    return f(0, 0, len(s1))


t = answer_scramble_tree("great", "rgeat")
print_tree(t)
assert apply_tree(t) == "rgeat" and tree_source(t) == "great"
assert answer_scramble_tree("abcde", "caebd") is None

rng = random.Random(5)
for _ in range(200):                                   # kiểm tra trên cặp ngẫu nhiên chắc chắn True
    s1 = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 10)))
    s2 = random_scramble(s1, rng)
    tree = answer_scramble_tree(s1, s2)
    assert apply_tree(tree) == s2 and tree_source(tree) == s1
print("Bài 5: PASS")

'great' = 'gr' | 'eat'  [giữ nguyên] -> 'rgeat'
    'gr' = 'g' | 'r'  [ĐỔI CHỖ] -> 'rg'
        'g'
        'r'
    'eat'
Bài 5: PASS


In [39]:
# ===== Đáp án Bài 6: đếm số cây xáo trộn =====
def answer_count_trees(s1: str, s2: str) -> int:
    @lru_cache(maxsize=None)
    def f(i1, i2, length):
        if length == 1:
            return 1 if s1[i1] == s2[i2] else 0
        if sorted(s1[i1:i1 + length]) != sorted(s2[i2:i2 + length]):
            return 0                                   # cắt tỉa này vẫn đúng khi đếm
        total = 0
        for k in range(1, length):
            total += f(i1, i2, k) * f(i1 + k, i2 + k, length - k)              # không đổi chỗ
            total += f(i1, i2 + length - k, k) * f(i1 + k, i2, length - k)     # đổi chỗ
        return total

    return f(0, 0, len(s1))


assert answer_count_trees("ab", "ba") == 1
assert answer_count_trees("aa", "aa") == 2
assert answer_count_trees("abc", "abc") == 2
assert answer_count_trees("abcd", "abcd") == 5
assert answer_count_trees("abcde", "caebd") == 0
print("great -> rgeat:", answer_count_trees("great", "rgeat"), "cây")
print("Catalan qua 'abcdefg'[:n], n=1..7:", [answer_count_trees("abcdefg"[:n], "abcdefg"[:n]) for n in range(1, 8)])

# đối chiếu: số cây > 0  <=>  là scramble
rng = random.Random(6)
for _ in range(300):
    s1, s2 = make_pair(rng.randint(1, 8), rng)
    assert (answer_count_trees(s1, s2) > 0) == Solution().isScramble(s1, s2)
print("Bài 6: PASS")

great -> rgeat: 5 cây
Catalan qua 'abcdefg'[:n], n=1..7: [1, 1, 2, 5, 14, 42, 132]


Bài 6: PASS


## 19. Template ngắn nên nhớ

```python
from functools import lru_cache

def is_scramble(s1, s2):
    @lru_cache(maxsize=None)
    def f(i1, i2, L):                       # s1[i1:i1+L] ~ s2[i2:i2+L] ?
        a, b = s1[i1:i1 + L], s2[i2:i2 + L]
        if a == b:               return True
        if sorted(a) != sorted(b): return False
        for k in range(1, L):               # điểm cắt: 2 phần KHÁC RỖNG
            if f(i1, i2, k) and f(i1 + k, i2 + k, L - k):          return True  # giữ nguyên
            if f(i1, i2 + L - k, k) and f(i1 + k, i2, L - k):      return True  # đổi chỗ
        return False
    return f(0, 0, len(s1))
```

Bottom-up: `dp[L][i][j]`, `L` tăng dần là vòng **ngoài cùng**:

```text
dp[1][i][j] = s1[i] == s2[j]
dp[L][i][j] = OR_k  (dp[k][i][j] and dp[L-k][i+k][j+k])      # giữ nguyên
                 or (dp[k][i][j+L-k] and dp[L-k][i+k][j])    # đổi chỗ
```

Câu thần chú: **"Nhìn nhát cắt đầu tiên: nửa dài `k` nằm ở `k` ký tự ĐẦU (giữ) hoặc `k` ký tự CUỐI (đổi) của `s2`."**

## 20. Checklist tự kiểm tra trước khi submit

- Điểm cắt có chạy đúng `range(1, n)` (hai phần **khác rỗng**) không?
- Trường hợp **đổi chỗ** có ghép đúng độ dài: nửa dài `k` của `s1` với `k` ký tự **cuối** của `s2` (`i2 + length - k`), nửa dài `length - k` với đầu `s2` (`i2`)?
- Có cắt tỉa `a == b → True` và "khác bộ chữ → False" không? (Nhớ: cùng bộ chữ **chưa đủ** để trả `True`.)
- Có memo không? (Không memo → hàm mũ, TLE với chuỗi lặp nhiều chữ.)
- Hàm memo có nằm **bên trong** `isScramble` để không dùng nhầm sổ của test trước?
- Với bản bottom-up: bảng tạo bằng list comprehension (không `* n` lồng nhau), vòng `L` ở **ngoài cùng** và tăng dần?
- `return True` ngay khi tìm thấy; `return False` chỉ **sau** khi thử hết mọi `k`?
- Đã test 3 ví dụ của đề + `"abcd"/"bdac"` (anagram nhưng không phải scramble) + chuỗi dài 30 nhiều chữ lặp chưa?